# 4. SupportSense — A/B Experiment
## Experiment design
- **Variant A:** current baseline behaviour—retrieve the top passage and return it as the answer.
- **Variant B:** RAG assistant—retrieve relevant evidence and apply the RAG answer policy, including a similarity threshold and abstention. If configured, it can use an LLM; otherwise it uses an extractive fallback.

**Important:** with the default extractive fallback, A and B can return the same top passage for answerable questions. Similar metrics are expected and must not be presented as proof of improvement. A genuine online A/B test also requires randomized assignment, sufficient traffic, and pre-specified outcome metrics.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from rag_core import SupportKnowledgeBase, baseline_answer, rag_answer, evaluate_answer

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
docs = json.loads((ROOT / "data" / "company_docs.json").read_text(encoding="utf-8"))
questions = json.loads((ROOT / "data" / "evaluation_questions.json").read_text(encoding="utf-8"))
kb = SupportKnowledgeBase(docs)

In [ ]:
records = []
for q in questions:
    retrieved = kb.retrieve(q["question"], k=3)
    for variant in ["A_baseline", "B_rag"]:
        start = time.perf_counter()
        result = baseline_answer(q["question"], retrieved) if variant == "A_baseline" else rag_answer(q, retrieved)
        latency_ms = (time.perf_counter() - start) * 1000
        records.append(evaluate_answer(q, result, retrieved, latency_ms, variant))
results = pd.DataFrame(records)
display(results.head(10))

In [ ]:
metric_cols = ["retrieval_hit", "grounded_answer_proxy", "citation_coverage",
               "correct_abstention", "unsupported_answer_proxy", "unnecessary_abstention",
               "latency_ms", "estimated_handle_seconds"]
summary = results.groupby("variant")[metric_cols].mean().round(3)
display(summary)

## How to interpret the experiment
- Retrieval hit rate measures whether the expected document was retrieved; both variants share the retriever.
- `grounded_answer_proxy` and unsupported-answer indicators are automated proxies based on synthetic labels.
- `estimated_handle_seconds` is a scenario assumption for business modelling, not observed agent time.
- The current script is an **offline paired benchmark**, not a live randomized A/B experiment.
- Before claiming a win, evaluate more representative questions, use blinded human grading, define a primary metric in advance, estimate sample size, and report uncertainty.